In [1]:
import os
import wfdb

# Generated patients: p00000 to p00039
patients_generated = [f"p{idx:05d}" for idx in range(41)]

# Manual patients
patients_manual = [
    'p00019', 'p00084', 'p00173', 'p00190', 'p00206',
    'p00214', 'p00244', 'p00261', 'p00283', 'p00284', 'p00380'
]

# Combine and remove duplicates
selected_patients = sorted(list(set(patients_generated + patients_manual)))

fs = 250
data_folder = "icentia_data"

NORMAL_BEATS = ['N']
ARRHYTHMIC_BEATS = ['S', 'V', 'Q']

beat_length = 150

num_train_beats = 500
num_val_beats = 100
num_test_norm = 100
num_test_arr = 100

required_normal = num_train_beats + num_val_beats + num_test_norm
required_arr = num_test_arr

def get_segments(signal, ann_samples, ann_symbols, beat_length, target_symbols):
    segments = []
    for s, sym in zip(ann_samples, ann_symbols):
        if sym in target_symbols and s >= beat_length:
            seg = signal[s - beat_length:s]
            segments.append(seg)
    return segments

# ===============================
# CHECK ALL PATIENTS
# ===============================
patients_insufficient = []

for patient in selected_patients:

    patient_folder = os.path.join(data_folder, "p00", patient)

    if not os.path.exists(patient_folder):
        print(f"{patient} → folder missing")
        continue

    records = sorted([
        f.replace(".hea", "")
        for f in os.listdir(patient_folder)
        if f.endswith(".hea")
    ])

    normal_segments = []
    arr_segments = []

    for rec_name in records:

        rec_path = os.path.join(patient_folder, rec_name)

        try:
            rec = wfdb.rdrecord(rec_path)
            ann = wfdb.rdann(rec_path, 'atr')
        except:
            continue

        signal = rec.p_signal[:, 0]

        norm_seg = get_segments(signal, ann.sample, ann.symbol, beat_length, NORMAL_BEATS)
        arr_seg  = get_segments(signal, ann.sample, ann.symbol, beat_length, ARRHYTHMIC_BEATS)

        normal_segments.extend(norm_seg)
        arr_segments.extend(arr_seg)

    n_norm = len(normal_segments)
    n_arr  = len(arr_segments)

    status = "OK"

    if n_norm < required_normal or n_arr < required_arr:
        status = "INSUFFICIENT"
        patients_insufficient.append(patient)

    print(f"{patient} → Normal: {n_norm}, Arr: {n_arr} → {status}")

print("\n==============================")
print("Patients with insufficient data:")
print(patients_insufficient)
print(f"Total insufficient: {len(patients_insufficient)}")

p00000 → Normal: 273060, Arr: 9610 → OK
p00001 → Normal: 190911, Arr: 67864 → OK
p00002 → Normal: 131227, Arr: 100369 → OK
p00003 → Normal: 91298, Arr: 162668 → OK
p00004 → Normal: 102376, Arr: 130445 → OK
p00005 → Normal: 140477, Arr: 56041 → OK
p00006 → Normal: 178583, Arr: 127743 → OK
p00007 → Normal: 223703, Arr: 25478 → OK
p00008 → Normal: 176178, Arr: 79119 → OK
p00009 → Normal: 80247, Arr: 169549 → OK
p00010 → Normal: 218737, Arr: 68332 → OK
p00011 → Normal: 232825, Arr: 27047 → OK
p00012 → Normal: 118412, Arr: 104261 → OK
p00013 → Normal: 255707, Arr: 18275 → OK
p00014 → Normal: 205574, Arr: 2312 → OK
p00015 → Normal: 210100, Arr: 42186 → OK
p00016 → Normal: 229175, Arr: 23696 → OK
p00017 → Normal: 106070, Arr: 175743 → OK
p00018 → Normal: 146665, Arr: 50406 → OK
p00019 → Normal: 225828, Arr: 40842 → OK
p00020 → Normal: 162027, Arr: 91161 → OK
p00021 → Normal: 229716, Arr: 75831 → OK
p00022 → Normal: 293708, Arr: 12087 → OK
p00023 → Normal: 246309, Arr: 41785 → OK
p00024 → Norm